# arms / heterogeneity — does the gain depend on who the patient is?  `[EVAL]`

Every eval metric split by **true persona trait** (recovered across the per-iteration persona shuffle via
`persona_id`, never `file_index`), for **all four arms** on one axis — `PTO_LA0`, `PTO_LA5`, `GRPO_LA0`,
`GRPO_LA5`. Traits: `cooperation_level` (Resistant / Cooperative / Warms up) and `problem` (Smoking / Obesity).

Three views per trait, in `results/arms/heterogeneity/figures/<judge>/`:
1. a **combined all-metrics overview** (`<trait>_all_metrics.png` — rows = metric, cols = arm),
2. the **per-metric subfolder** (`<trait>/<metric>.png` — one panel per arm), and
3. **endpoint bars** at each arm's **final AND best** iteration (`subgroup_endpoint_<trait>_{final,best}.png`),
   backed by a per-trait table of the subgroup means (`tables/<judge>/subgroup_endpoint_means_<trait>.md`).

**Grader.** Per-judge family: this notebook is rendered once per grader (`EDA_JUDGE`) — the primary oracle
(gpt-4o-mini, the training reward) and the held-out judge (claude-haiku-4-5) — so every artifact names the grader
that produced it. **Best iteration** = each arm's peak Q1+Q2 (its own training rubric) *under the grader being
rendered*: on the primary leaf that is own-oracle selection; on the held-out leaf it is the held-out judge's pick,
so the selected iterations can differ between the two leaves (each caption lists them).
**Sign convention:** higher = better for every rubric except `MICI` (↓ suffix, lower = better). **Unit:** the 96
conversations of one model state split by trait category (disjoint persona subsets, no pairing). **Support:**
every arm's trajectory stops at its own last scored state under the grader being rendered, and its *final* bars sit there (the iteration itself is derived per render and stated per caption, and the note is empty when no arm is short).


In [ ]:
import sys, os
_p = os.path.abspath(".")                      # find eda/ (the dir holding eda_analysis/) from any depth
while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "eda_analysis")):
    _p = os.path.dirname(_p)
sys.path.insert(0, _p)
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 185, "display.max_columns", 50)

import os, eda_analysis
from eda_analysis import exports, plotting
from eda_analysis.constants import judge_dirname
# FAMILY = which results folder this notebook owns; JUDGE = which grader's scores are read
# ("" = the primary oracle; render_results.py sets EDA_JUDGE per grader on disk).
cfg = eda_analysis.EdaConfig(family="arms/heterogeneity", judge=os.environ.get("EDA_JUDGE", ""))
S = eda_analysis.notebook_setup(cfg)
exports.reset_results()   # clears only THIS family's generated figures/tables for the active judge (never SUMMARY.md)
exports.save_provenance(cfg, S.SCORES)   # re-stamp the leaf's _provenance.md (reset just removed the one notebook_setup wrote)

TRAITS = ["cooperation_level", "problem"]      # persona traits to split by (see PERSONA_COLS for more)
GRADER = judge_dirname(S.JUDGE)                # short grader label named in every caption (gpt-4o-mini | claude-haiku-4-5)
GRADER_KIND = "primary oracle = the training reward" if not S.JUDGE else "held-out judge, not the training reward"
# Best iteration per arm = peak Q1+Q2 (own training rubric) UNDER THIS GRADER — own-oracle selection on the
# primary leaf, the held-out judge's pick on the held-out leaf. Drives the *_best endpoint bars/tables.
BEST = eda_analysis.best_iteration_by_arm(S.SCORES)
FINAL = {a: int(g.loc[~g.is_base, "iteration"].max()) for a, g in S.SCORES.groupby("arm")}
BEST_BY = f"peak Q1+Q2 (own training rubric) under {GRADER}"
# This inline derivation is now the package helper (constants.support_note), so every family
# states censoring the same way and no family can drift back to a written-down iteration.
CENSOR = eda_analysis.support_note(S.SCORES, prefix="", label=False,
                                   subject=f"no later state scored by {GRADER}")
CENSOR_NOTE = f" Censoring: {CENSOR}" if CENSOR else ""
SIGN_NOTE = "Higher = better for every rubric except MICI (↓, lower = better)."
UNIT_NOTE = ("Unit = the 96 conversations of one model state, split by the true persona trait (categories are "
             "disjoint persona subsets, no pairing); mean per category with a 95% bootstrap CI (seed BOOT_SEED).")
print(f"grader: {GRADER} ({GRADER_KIND})")
print("final iteration per arm:", FINAL)
print("best iteration per arm: ", BEST, "<-", BEST_BY)


## 1 · Combined overview — all metrics per trait  `[EVAL]`
The single-glance companion to the per-metric subfolder below: one figure per trait with **rows = metric,
cols = arm** (all four arms), each cell split by persona category. Mirrors `arms/outcomes` `trajectories_all_metrics`.
→ `figures/<judge>/<trait>_all_metrics.png`.

In [ ]:
for trait in TRAITS:
    figo = plotting.heterogeneity_overview_grid(S.SCORES, trait, arms=cfg.focus_arms, metrics=S.METRICS)
    if figo is None:
        print(f"{trait}: nothing plottable."); continue
    exports.save_fig(
        figo, f"{trait}_all_metrics",
        caption=(f"[{GRADER}; {GRADER_KIND}] All metrics by true patient {trait} across iterations (rows = metric, "
                 f"cols = arm; all four arms), each cell split by persona category — the combined overview companion "
                 f"to the per-metric {trait}/ subfolder. {SIGN_NOTE} {UNIT_NOTE}{CENSOR_NOTE}"))
    plt.show()


## 2 · Every metric × every trait  `[EVAL]`
One small-multiples figure per (trait, metric): the metric across iterations, split by trait category (readable
names, colourblind palette), a panel per arm. → `figures/<judge>/<trait>/<metric>.png` (nested groups
`cooperation_level/`, `problem/`).

In [ ]:
for trait in TRAITS:
    for m in S.METRICS:
        fig = plotting.heterogeneity_grid(S.SCORES, trait, arms=cfg.focus_arms, metric=m)
        if fig is None:
            print(f"{trait} x {m}: nothing plottable."); continue
        exports.save_fig(
            fig, m, group=trait,
            caption=(f"[{GRADER}; {GRADER_KIND}] {eda_analysis.display_label(m)} across iterations split by true "
                     f"patient {trait}; one panel per arm (all four arms). {SIGN_NOTE} {UNIT_NOTE}{CENSOR_NOTE}"))
        plt.show()


## 3 · Endpoint bars — where does the gap concentrate? FINAL + BEST  `[EVAL]`
Score per trait category × arm at each arm's **final** and **best** iteration (best = peak Q1+Q2 under the grader
being rendered; dotted = pooled base over all arms and categories). **Read (primary leaf):** GRPO's late regression
concentrates on the *Resistant* (Low-cooperation) personas while PTO holds above base — and the final-vs-best pair
shows how much of that subgroup damage is post-peak. The table beneath each pair records the numbers behind the bars
for every metric (n, mean, sd, sem per category × arm × target, plus the arm's own iteration-0 mean for that category
and the delta to it).
→ `figures/<judge>/subgroup_endpoint_<trait>_{final,best}.png`, `tables/<judge>/subgroup_endpoint_means_<trait>.md`.


In [ ]:
def subgroup_endpoint_means(scores_long, trait, metrics, *, targets):
    # The numbers behind subgroup_endpoint_bars: mean per (metric, target, arm, category) at the arm's
    # target iteration, plus the arm's own iteration-0 mean for that category and the delta to it.
    rows = []
    for m in metrics:
        d = scores_long[(scores_long.questionnaire == m) & scores_long[trait].notna()]
        for arm in sorted(d.arm.unique()):
            da = d[d.arm == arm]
            base = da[da.is_base].groupby(trait)["score"].mean()
            for tgt_label, tmap in targets:
                it = int((tmap or {}).get(arm, da.iteration.max()))
                de = da[da.iteration == it]
                for cat, g in de.groupby(trait):
                    rows.append({"metric": m, "target": tgt_label, "arm": arm, "iteration": it,
                                 trait: cat, "n": int(len(g)), "mean": g.score.mean(),
                                 "sd": g.score.std(ddof=1), "sem": g.score.sem(),
                                 "base_mean": base.get(cat, np.nan),
                                 "delta_vs_base": g.score.mean() - base.get(cat, np.nan)})
    return pd.DataFrame(rows)

TARGETS = (("final", None), ("best", BEST))
BEST_LIST = ", ".join(f"{a} = iter {i}" for a, i in sorted(BEST.items()))
FINAL_LIST = ", ".join(f"{a} = iter {i}" for a, i in sorted(FINAL.items()))
for trait in TRAITS:
    for tgt, tmap, note in (("final", None, f"final iteration ({FINAL_LIST})"),
                            ("best", BEST, f"best iteration = {BEST_BY} ({BEST_LIST})")):
        figb = plotting.subgroup_endpoint_bars(S.SCORES, trait, arms=cfg.focus_arms,
                                               metric=cfg.focus_metric, palette=S.PALETTE,
                                               iter_by_arm=tmap, target_label=tgt)
        if figb is None:
            print(f"{trait} ({tgt}): nothing plottable."); continue
        exports.save_fig(
            figb, f"subgroup_endpoint_{trait}_{tgt}",
            caption=(f"[{GRADER}; {GRADER_KIND}] {eda_analysis.display_label(cfg.focus_metric)} by true patient {trait} "
                     f"at each arm's {note}, grouped bars per arm (all four arms; dotted = pooled base over all arms) "
                     f"— where the endpoint gap concentrates. {SIGN_NOTE} {UNIT_NOTE}{CENSOR_NOTE}"))
        plt.show()
    tab = subgroup_endpoint_means(S.SCORES, trait, S.METRICS, targets=TARGETS)
    exports.save_table(
        tab, f"subgroup_endpoint_means_{trait}",
        caption=(f"[{GRADER}; {GRADER_KIND}] The numbers behind the subgroup_endpoint_{trait}_* bars, for every metric: "
                 f"mean / sd / sem / n per (metric, target ∈ {{final, best}}, arm, {trait} category) at the arm's target "
                 f"iteration, plus the arm's OWN iteration-0 mean for that category (base_mean) and delta_vs_base = "
                 f"mean − base_mean (⇒ trained model higher; for MICI ↓ negative = improvement). "
                 f"Best = {BEST_BY} ({BEST_LIST}); final = last scored iteration ({FINAL_LIST}). "
                 f"{SIGN_NOTE} {UNIT_NOTE}{CENSOR_NOTE}"))
    display(tab[tab.metric == cfg.focus_metric].round(3))


## 4 · Artifact index
Drop stale caption lines and refresh `results/arms/INDEX.md` + the root `results/INDEX.md` (every notebook ends with this).


In [ ]:
print("pruned captions:", exports.prune_orphan_captions())
print("index ->", exports.build_index())
